In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
CSV_PATH = "../data/Trade-off Method (RFTM) new.csv"

In [3]:
df = pd.read_csv(CSV_PATH, decimal=",")
df.columns = [c.strip() for c in df.columns]

In [4]:
FEATURES = ["x[1]", "x[2]", "x[3]", "x[4]", "x[5]", "x[6]",
            "gender", "age", "Sub-experiment"]
TARGETS = ["p[1]", "p[2]", "p[3]", "p[4]", "p[5]"]

In [5]:
X= df[FEATURES].to_numpy(dtype=float)
y = df[TARGETS].to_numpy(dtype=float)

In [6]:
print(f"Ornek sayisi: {X.shape[0]}, ozellik: {X.shape[1]}, hedef: {y.shape[1]}")


Ornek sayisi: 110, ozellik: 9, hedef: 5


In [20]:
def build_model(seed: int = 42):
    mlp = MLPRegressor(
        hidden_layer_sizes=(8,),   
        activation="relu",
        alpha=10.0,            
        learning_rate_init=1e-3,
        max_iter=5000,
        early_stopping=False,  
        random_state=seed,
    )
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), mlp),
        transformer=StandardScaler(),
    )
 

In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [10]:
def build_ridge():
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(),
                                RidgeCV(alphas=np.logspace(-3, 3, 20))),
        transformer=StandardScaler(),
    )

In [11]:
models = {
    "ANN (MLP)": build_model,
    "Ridge": build_ridge,
    "Baseline (ortalama)": lambda: DummyRegressor(strategy="mean"),
}

In [12]:
mae = {name: np.zeros((kf.get_n_splits(), y.shape[1])) for name in models}


In [13]:
for fold, (tr, te) in enumerate(kf.split(X)):
    for name, make in models.items():
        pred = make().fit(X[tr], y[tr]).predict(X[te])
        for j in range(y.shape[1]):
            mae[name][fold, j] = mean_absolute_error(y[te, j], pred[:, j])

In [14]:
print("\n=== 5-Fold CV, MAE (dusuk = iyi) ===")
print(f"{'hedef':<8}" + "".join(f"{n:>22}" for n in models))
for j, t in enumerate(TARGETS):
    print(f"{t:<8}" + "".join(f"{mae[n][:, j].mean():>22.4f}" for n in models))
print(f"{'ORTALAMA':<8}" + "".join(f"{mae[n].mean():>22.4f}" for n in models))


=== 5-Fold CV, MAE (dusuk = iyi) ===
hedef                ANN (MLP)                 Ridge   Baseline (ortalama)
p[1]                    0.0313                0.0305                0.0345
p[2]                    0.0664                0.0659                0.0679
p[3]                    0.0796                0.0785                0.0861
p[4]                    0.0769                0.0768                0.0803
p[5]                    0.0292                0.0292                0.0296
ORTALAMA                0.0567                0.0562                0.0597


In [15]:
final_model = build_model().fit(X, y)
print("\nSon model egitildi (tum veri).")


Son model egitildi (tum veri).
